# AI Framing Classification Pipeline

This notebook constructs a transparent rule-based AI sentence framing dataset for earnings calls. It:

1. Loads the call-level AI dataset and transcript full text.
2. Matches transcript text to each call using identifiers and fallback composite keys.
3. Screens for AI-related sentences using a narrow AI keyword list with strict boundaries.
4. Classifies each AI-related sentence into one dominant framing category: opportunity, implementation, risk, or mixed/unclear.
5. Aggregates sentence-level labels to call-level framing intensity and share variables.
6. Saves the sentence-level classifications, validation sample, and expanded call-level dataset.

In [ ]:
from pathlib import Path
import re
import json
import numpy as np
import pandas as pd

pd.set_option('display.max_columns', 100)
pd.set_option('display.max_colwidth', 160)

PROJECT_ROOT = Path.cwd().resolve()
DATA_PROCESSED = PROJECT_ROOT / 'data' / 'processed'
DATA_RAW = PROJECT_ROOT / 'data' / 'raw'
OUTPUT_TABLES = PROJECT_ROOT / 'output' / 'tables'
OUTPUT_TABLES.mkdir(parents=True, exist_ok=True)

CALL_LEVEL_PATH = DATA_PROCESSED / 'earnings_call_ai_dataset.csv'
REQUESTED_TRANSCRIPT_PATH = DATA_RAW / 'earnings_call_transcripts_2019_2025_full_text.csv'
FALLBACK_TRANSCRIPT_PATH = DATA_RAW / 'earnings_call_transcripts_2019_2025_with_text.csv'

SENTENCE_OUTPUT_PATH = OUTPUT_TABLES / 'ai_sentence_classification.csv'
VALIDATION_OUTPUT_PATH = OUTPUT_TABLES / 'framing_validation_sample.csv'
FRAMING_DATASET_OUTPUT_PATH = DATA_PROCESSED / 'earnings_call_ai_framing_dataset.csv'

print(f'Project root: {PROJECT_ROOT}')
print(f'Call-level input: {CALL_LEVEL_PATH}')
print(f'Requested transcript input: {REQUESTED_TRANSCRIPT_PATH}')

## 1. Load call-level data and transcript text

In [ ]:
call_df = pd.read_csv(CALL_LEVEL_PATH)

if REQUESTED_TRANSCRIPT_PATH.exists():
    transcript_path = REQUESTED_TRANSCRIPT_PATH
    transcript_source_note = 'requested path'
elif FALLBACK_TRANSCRIPT_PATH.exists():
    transcript_path = FALLBACK_TRANSCRIPT_PATH
    transcript_source_note = 'fallback path because requested *_full_text.csv was not present'
else:
    raise FileNotFoundError(
        'Could not find transcript text file at either '
        f'{REQUESTED_TRANSCRIPT_PATH} or {FALLBACK_TRANSCRIPT_PATH}'
    )

transcript_df = pd.read_csv(transcript_path)

print(f'Loaded call-level rows: {len(call_df):,}')
print(f'Loaded transcript rows: {len(transcript_df):,}')
print(f'Transcript source used: {transcript_path.name} ({transcript_source_note})')
print('\nCall-level columns:')
print(list(call_df.columns))
print('\nTranscript columns:')
print(list(transcript_df.columns))

## 2. Match full transcript text to call-level dataset

The matching logic prioritizes stable identifiers, then uses composite fallback keys where identifiers are missing.

In [ ]:
def normalize_id_series(s: pd.Series) -> pd.Series:
    """Normalize numeric-looking identifiers to stable string keys."""
    return pd.to_numeric(s, errors='coerce').astype('Int64').astype(str).replace('<NA>', pd.NA)


def normalize_text_key(s: pd.Series) -> pd.Series:
    return (
        s.astype('string')
        .str.lower()
        .str.replace(r'\s+', ' ', regex=True)
        .str.strip()
    )


def normalize_date_key(s: pd.Series) -> pd.Series:
    return pd.to_datetime(s, errors='coerce').dt.strftime('%Y-%m-%d').astype('string')

call = call_df.copy()
transcripts = transcript_df.copy()

for df in [call, transcripts]:
    if 'transcriptid' in df.columns:
        df['_transcriptid_key'] = normalize_id_series(df['transcriptid'])
    else:
        df['_transcriptid_key'] = pd.NA
    if 'keydevid' in df.columns:
        df['_keydevid_key'] = normalize_id_series(df['keydevid'])
    else:
        df['_keydevid_key'] = pd.NA
    df['_ticker_key'] = normalize_text_key(df['ticker']) if 'ticker' in df.columns else pd.NA
    df['_headline_key'] = normalize_text_key(df['headline']) if 'headline' in df.columns else pd.NA
    df['_call_date_key'] = normalize_date_key(df['call_date']) if 'call_date' in df.columns else pd.NA

text_col_candidates = ['transcript_text', 'full_text', 'text', 'transcript']
text_col = next((c for c in text_col_candidates if c in transcripts.columns), None)
if text_col is None:
    raise ValueError(f'No transcript text column found. Checked: {text_col_candidates}')

# Deduplicate transcript source by the strongest available keys before merging.
transcript_cols_to_keep = [
    c for c in [
        '_transcriptid_key', '_keydevid_key', '_ticker_key', '_headline_key', '_call_date_key',
        text_col, 'transcriptcollectiontypename', 'companyid', 'sector', 'sub_industry', 'security'
    ] if c in transcripts.columns
]
transcript_lookup = transcripts[transcript_cols_to_keep].copy()
transcript_lookup = transcript_lookup.rename(columns={text_col: 'transcript_text_full'})

call['_row_id'] = np.arange(len(call))
matched = call.copy()
matched['transcript_text_full'] = pd.NA
matched['text_match_method'] = pd.NA

match_steps = [
    ('transcriptid', ['_transcriptid_key']),
    ('keydevid', ['_keydevid_key']),
    ('transcriptid_keydevid', ['_transcriptid_key', '_keydevid_key']),
    ('ticker_date_headline', ['_ticker_key', '_call_date_key', '_headline_key']),
    ('date_headline', ['_call_date_key', '_headline_key']),
]

match_counts = []
for method, keys in match_steps:
    usable = all(k in transcript_lookup.columns for k in keys)
    if not usable:
        match_counts.append({'method': method, 'new_matches': 0, 'note': 'keys unavailable'})
        continue
    source = transcript_lookup.dropna(subset=keys + ['transcript_text_full']).copy()
    if source.empty:
        match_counts.append({'method': method, 'new_matches': 0, 'note': 'no usable source rows'})
        continue
    source = source.sort_values(keys).drop_duplicates(subset=keys, keep='first')
    source = source[keys + ['transcript_text_full']]
    unresolved_ids = matched.loc[matched['transcript_text_full'].isna(), ['_row_id'] + keys]
    unresolved_ids = unresolved_ids.dropna(subset=keys)
    if unresolved_ids.empty:
        match_counts.append({'method': method, 'new_matches': 0, 'note': 'no unresolved rows with required keys'})
        continue
    step = unresolved_ids.merge(source, on=keys, how='left')
    step = step.dropna(subset=['transcript_text_full'])
    if step.empty:
        match_counts.append({'method': method, 'new_matches': 0, 'note': 'no matches'})
        continue
    new_text = step.set_index('_row_id')['transcript_text_full']
    new_idx = new_text.index
    matched.loc[matched['_row_id'].isin(new_idx), 'transcript_text_full'] = matched.loc[
        matched['_row_id'].isin(new_idx), '_row_id'
    ].map(new_text)
    matched.loc[matched['_row_id'].isin(new_idx), 'text_match_method'] = method
    match_counts.append({'method': method, 'new_matches': len(new_idx), 'note': 'matched'})

match_diag = pd.DataFrame(match_counts)
matched_count = matched['transcript_text_full'].notna().sum()
print('Match diagnostics by step:')
display(match_diag)
print(f"Total call-level rows: {len(matched):,}")
print(f"Rows matched with transcript text: {matched_count:,} ({matched_count / len(matched):.2%})")
print(f"Rows unmatched with transcript text: {len(matched) - matched_count:,}")
print('\nMatch method distribution:')
print(matched['text_match_method'].fillna('unmatched').value_counts(dropna=False))

## 3. Identify AI-related sentences using narrow keyword screen

Broad technology terms are intentionally excluded from the AI sentence screen. They are allowed only as context clues inside the framing classifier after the narrow AI screen has already matched a sentence.

In [ ]:
AI_TERM_PATTERNS = [
    ('artificial intelligence', r'\bartificial\s+intelligence\b'),
    ('generative AI', r'\bgenerative\s+AI\b'),
    ('GenAI', r'\bGenAI\b'),
    ('machine learning', r'\bmachine\s+learning\b'),
    ('deep learning', r'\bdeep\s+learning\b'),
    ('natural language processing', r'\bnatural\s+language\s+processing\b'),
    ('large language models', r'\blarge\s+language\s+models\b'),
    ('large language model', r'\blarge\s+language\s+model\b'),
    ('foundation models', r'\bfoundation\s+models\b'),
    ('foundation model', r'\bfoundation\s+model\b'),
    ('ChatGPT', r'\bChatGPT\b'),
    ('GPT-4', r'\bGPT-4\b'),
    ('GPT-5', r'\bGPT-5\b'),
    ('GPT', r'\bGPT(?:-\d+)?\b'),
    ('OpenAI', r'\bOpenAI\b'),
    ('Copilot', r'\bCopilot\b'),
    ('Gemini', r'\bGemini\b'),
    ('Claude', r'\bClaude\b'),
    ('Anthropic', r'\bAnthropic\b'),
    ('LLMs', r'\bLLMs\b'),
    ('LLM', r'\bLLM\b'),
    ('AI', r'\bAI\b'),
]

AI_REGEXES = [(term, re.compile(pattern, flags=re.IGNORECASE)) for term, pattern in AI_TERM_PATTERNS]

ABBREVIATION_PROTECT = {
    'Mr.': 'Mr<prd>', 'Ms.': 'Ms<prd>', 'Mrs.': 'Mrs<prd>', 'Dr.': 'Dr<prd>',
    'Prof.': 'Prof<prd>', 'Inc.': 'Inc<prd>', 'Ltd.': 'Ltd<prd>', 'Co.': 'Co<prd>',
    'Corp.': 'Corp<prd>', 'U.S.': 'U<prd>S<prd>', 'U.K.': 'U<prd>K<prd>',
    'e.g.': 'e<prd>g<prd>', 'i.e.': 'i<prd>e<prd>', 'vs.': 'vs<prd>'
}


def split_sentences(text):
    if pd.isna(text):
        return []
    text = str(text)
    text = re.sub(r'\s+', ' ', text).strip()
    if not text:
        return []
    for original, protected in ABBREVIATION_PROTECT.items():
        text = text.replace(original, protected)
    # Protect decimal points.
    text = re.sub(r'(?<=\d)\.(?=\d)', '<prd>', text)
    parts = re.split(r'(?<=[.!?])\s+(?=[A-Z0-9"\'])', text)
    sentences = []
    for part in parts:
        part = part.replace('<prd>', '.').strip()
        if part:
            sentences.append(part)
    return sentences


def matched_ai_terms(sentence):
    terms = []
    for term, regex in AI_REGEXES:
        if regex.search(sentence):
            terms.append(term)
    # Remove generic duplicates when a more specific term already matched.
    term_set = set(terms)
    if 'GPT' in term_set and any(t in term_set for t in ['GPT-4', 'GPT-5']):
        term_set.discard('GPT')
    if 'AI' in term_set and any(t.lower().endswith(' ai') or t.lower() == 'genai' for t in term_set if t != 'AI'):
        # Keep standalone AI only when it is itself meaningful and not merely inside a longer matched phrase.
        if not re.search(r'\bAI\b', re.sub(r'\bgenerative\s+AI\b', '', sentence, flags=re.IGNORECASE)):
            term_set.discard('AI')
    ordered = [term for term, _ in AI_TERM_PATTERNS if term in term_set]
    return ordered

# Quick boundary tests for the narrow AI screen.
boundary_tests = {
    'AI standalone': matched_ai_terms('We are investing in AI capabilities.'),
    'said should not match AI': matched_ai_terms('The CEO said margins improved.'),
    'GPT standalone': matched_ai_terms('GPT is embedded in the product.'),
    'GPT-number': matched_ai_terms('We tested GPT-4 and GPT-5.'),
    'LLM standalone': matched_ai_terms('The LLM improves search.'),
    'LLMs standalone': matched_ai_terms('We evaluate multiple LLMs.'),
    'automation only': matched_ai_terms('Automation improved productivity.'),
}
print(json.dumps(boundary_tests, indent=2))

## 4. Rule-based semantic scoring for AI framing

The classifier scores a small window around each AI sentence: previous sentence, focal sentence, and next sentence. The focal AI sentence receives higher weight; adjacent context receives lower weight. This keeps the rule transparent while allowing sentence-level context to inform the dominant framing label.

In [ ]:
# Framing dictionaries are intentionally conservative. Terms are grouped into:
# - high-confidence focal anchors that can justify a dominant label;
# - secondary terms that add context but should not force a label alone.
FRAMING_PATTERNS = {
    'opportunity': {
        'anchor': [
            r'\bopportunit(?:y|ies)\b', r'\bgrowth\b', r'\bgrow(?:th|ing)?\b', r'\brevenue\b',
            r'\bdemand\b', r'\bcustomer value\b', r'\bcompetitive advantage\b', r'\bfuture upside\b',
            r'\bupside\b', r'\binnovation\b', r'\binnovative\b', r'\bnew product\b',
            r'\bmarket share\b', r'\bmoneti[sz](?:e|ation)\b', r'\bcommercial opportunity\b',
            r'\bvalue creation\b', r'\bstrategic opportunity\b', r'\btransformative opportunity\b',
            r'\bunlock(?:ing)? value\b', r'\bcustomer benefit(?:s)?\b'
        ],
        'secondary': [
            r'\bexpand\b', r'\bexpansion\b', r'\bstrategic\b', r'\bstrategy\b',
            r'\bdifferentiat(?:e|ion|ed)\b', r'\bleadership\b', r'\bsales\b', r'\bcommercial\b',
            r'\bproduct improvement\b', r'\benhance(?:d|ment|s)?\b', r'\bimprov(?:e|ed|ement|ing)\b',
            r'\bfuture\b', r'\bnext generation\b', r'\baccelerat(?:e|ed|ing|ion)\b', r'\bscale\b',
            r'\bunlock\b', r'\bwin\b', r'\bwinner\b', r'\bbenefit(?:s)?\b'
        ]
    },
    'implementation': {
        'anchor': [
            r'\bdeploy(?:ed|ing|ment)?\b', r'\broll(?:ed)? out\b', r'\blaunch(?:ed|ing)?\b',
            r'\bintegrat(?:e|ed|ing|ion)\b', r'\bimplement(?:ed|ing|ation)?\b',
            r'\badopt(?:ed|ing|ion)?\b', r'\busing\b', r'\buse case(?:s)?\b', r'\bused to\b',
            r'\bapplication(?:s)?\b', r'\bworkflow(?:s)?\b', r'\bproductivity\b',
            r'\befficien(?:cy|cies|t)\b', r'\bautomation\b', r'\bautomate(?:d|s|ing)?\b',
            r'\bcustomer service\b', r'\bcontact center\b', r'\binternal operation(?:s)?\b',
            r'\bpilot(?:s|ed|ing)?\b', r'\bin production\b', r'\bAI-powered\b',
            r'\bAI enabled\b', r'\bAI-enabled\b', r'\bsecurity product\b', r'\bcybersecurity solution\b'
        ],
        'secondary': [
            r'\buse(?:d|s)?\b', r'\bplatform\b', r'\btool(?:s)?\b', r'\bsolution(?:s)?\b',
            r'\bcapabilit(?:y|ies)\b', r'\bprocess(?:es)?\b', r'\boperation(?:s|al)?\b',
            r'\binternal\b', r'\bsupport\b', r'\boptimi[sz](?:e|ed|ation|ing)\b',
            r'\bcost savings?\b', r'\btraining\b', r'\bprototype\b', r'\bproduction\b',
            r'\broadmap\b', r'\binfrastructure\b', r'\bdata pipeline\b', r'\bmodel(?:s)?\b',
            r'\bengineering\b', r'\bdeveloper(?:s)?\b', r'\bagent(?:s|ic)?\b'
        ]
    },
    'risk': {
        'anchor': [
            r'\bAI risk(?:s)?\b', r'\brisk(?:s)?\s+(?:from|of|around|related to|associated with)\s+(?:AI|artificial intelligence|generative AI|GenAI|LLM|LLMs|large language model(?:s)?)\b',
            r'\b(?:AI|artificial intelligence|generative AI|GenAI|LLM|LLMs|large language model(?:s)?)\s+risk(?:s)?\b',
            r'\bregulat(?:e|ed|ion|ory|ions)\b', r'\bcompliance\b', r'\bprivacy\b',
            r'\bdata protection\b', r'\bgovernance\b', r'\bmodel governance\b', r'\bresponsible AI\b',
            r'\blegal risk(?:s)?\b', r'\blegal concern(?:s)?\b', r'\bliabilit(?:y|ies)\b',
            r'\bethic(?:s|al)\b', r'\bbias\b', r'\bhallucinat(?:e|ion|ions)\b',
            r'\binaccurate output(?:s)?\b', r'\binaccuracy\b', r'\baccuracy risk(?:s)?\b',
            r'\bsafety concern(?:s)?\b', r'\bAI safety\b', r'\bguardrail(?:s)?\b',
            r'\bcybersecurity threat(?:s)?\b', r'\bsecurity threat(?:s)?\b', r'\battack(?:s|ed|ing)?\b',
            r'\bvulnerabilit(?:y|ies)\b', r'\bdata breach(?:es)?\b', r'\bimplementation challenge(?:s)?\b',
            r'\bchallenges?\s+(?:implementing|deploying|integrating|adopting)\s+(?:AI|artificial intelligence|generative AI|GenAI)\b',
            r'\b(?:AI|artificial intelligence|generative AI|GenAI|LLM|LLMs|large language model(?:s)?)\s+uncertain(?:ty|ties)?\b', r'\buncertain(?:ty|ties)?\s+(?:around|about|over|with|regarding|related to)\s+(?:AI|artificial intelligence|generative AI|GenAI|LLM|LLMs|large language model(?:s)?)\b'
        ],
        'secondary': [
            r'\bdisclosure\b', r'\baudit(?:s|ed|ing)?\b', r'\bcontrol(?:s)?\b',
            r'\boperational risk(?:s)?\b', r'\bmodel risk(?:s)?\b', r'\blitigation\b',
            r'\blawsuit(?:s)?\b', r'\bintellectual property risk(?:s)?\b', r'\bIP risk(?:s)?\b'
        ]
    }
}

COMPILED_FRAMING = {
    category: {
        strength: [(pattern, re.compile(pattern, flags=re.IGNORECASE)) for pattern in patterns]
        for strength, patterns in groups.items()
    }
    for category, groups in FRAMING_PATTERNS.items()
}

CATEGORY_WEIGHTS = {'focal': 1.0, 'context': 0.45}
ANCHOR_WEIGHT = 1.0
SECONDARY_WEIGHT = 0.5
MIN_DOMINANT_SCORE = 1.0
MIN_DOMINANCE_MARGIN = 0.3
VERY_CLEAR_SCORE = 2.0

# Vague phrases that should not be treated as opportunity without concrete opportunity language.
VAGUE_AI_MENTION_RE = re.compile(
    r'^\s*(?:turning to|on|about|regarding|with respect to)?\s*(?:AI|artificial intelligence|generative AI|GenAI)\s*[,.:;!?-]*\s*$|'
    r'\b(?:AI|artificial intelligence|generative AI|GenAI)\s+is\s+(?:important|interesting|topical|a topic|an area)\b|'
    r'\b(?:views?|thoughts?|perspective|perspectives)\s+on\s+(?:AI|artificial intelligence|generative AI|GenAI)\b',
    flags=re.IGNORECASE
)

# Broad risk-ish words that are allowed as context only, never by themselves.
BROAD_RISK_ONLY_RE = re.compile(r'\b(?:security|IP|safety|challenge(?:s|d)?|control(?:s)?|disrupt(?:ion|ive|ed)?)\b', flags=re.IGNORECASE)
EXPLICIT_RISK_RE = re.compile('|'.join(FRAMING_PATTERNS['risk']['anchor']), flags=re.IGNORECASE)
SECURITY_PRODUCT_RE = re.compile(r'\b(?:AI-powered|AI enabled|AI-enabled|machine learning|artificial intelligence).{0,80}\b(?:security|cybersecurity|threat detection|fraud detection|breach detection|protection|defense|defence|solution|product|platform)\b', flags=re.IGNORECASE)
SECURITY_RISK_RE = re.compile(r'\b(?:cybersecurity threat(?:s)?|security threat(?:s)?|attack(?:s|ed|ing)?|vulnerabilit(?:y|ies)|data breach(?:es)?|privacy|governance|(?:AI|artificial intelligence|generative AI|GenAI|LLM|LLMs|large language model(?:s)?)\s+uncertain(?:ty|ties)?|uncertain(?:ty|ties)?\s+(?:around|about|over|with|regarding|related to)\s+(?:AI|artificial intelligence|generative AI|GenAI|LLM|LLMs|large language model(?:s)?))\b', flags=re.IGNORECASE)


def score_category(text, category):
    matched = []
    score = 0.0
    for strength, weight in [('anchor', ANCHOR_WEIGHT), ('secondary', SECONDARY_WEIGHT)]:
        for pattern, regex in COMPILED_FRAMING[category][strength]:
            if regex.search(text):
                score += weight
                matched.append(f'{strength}:{pattern}')
    return score, matched


def score_category_with_context(prev_sentence, sentence, next_sentence, category):
    focal_score, focal_terms = score_category(sentence, category)
    prev_score, prev_terms = score_category(prev_sentence or '', category)
    next_score, next_terms = score_category(next_sentence or '', category)
    total_score = (
        CATEGORY_WEIGHTS['focal'] * focal_score
        + CATEGORY_WEIGHTS['context'] * (prev_score + next_score)
    )
    matched = list(focal_terms)
    matched.extend([f'context:{t}' for t in prev_terms + next_terms])
    return total_score, focal_score, matched


def has_anchor(sentence, category):
    return any(regex.search(sentence) for _, regex in COMPILED_FRAMING[category]['anchor'])


def is_security_product_without_explicit_risk(sentence):
    return bool(SECURITY_PRODUCT_RE.search(sentence)) and not bool(SECURITY_RISK_RE.search(sentence))


def classify_ai_sentence(prev_sentence, sentence, next_sentence):
    score_payload = {}
    focal_scores = {}
    matched_terms = {cat: [] for cat in FRAMING_PATTERNS}

    for category in FRAMING_PATTERNS:
        total, focal, terms = score_category_with_context(prev_sentence, sentence, next_sentence, category)
        score_payload[category] = total
        focal_scores[category] = focal
        matched_terms[category].extend(terms)

    # Explicit security-product override: product/solution language without explicit threat, privacy,
    # governance, vulnerability, or uncertainty should be implementation rather than risk.
    if is_security_product_without_explicit_risk(sentence):
        score_payload['implementation'] = max(score_payload['implementation'], 1.2)
        score_payload['risk'] = 0.0
        matched_terms['implementation'].append('override:AI security product/solution without explicit risk language')
        matched_terms['risk'].append('suppressed:broad security product language')

    # Broad risk-adjacent words should not create risk labels without explicit AI-risk language.
    if score_payload['risk'] > 0 and not EXPLICIT_RISK_RE.search(sentence):
        # Preserve a small contextual score for diagnostics, but prevent broad/context-only risk from winning.
        if BROAD_RISK_ONLY_RE.search(sentence) or focal_scores['risk'] == 0:
            score_payload['risk'] = min(score_payload['risk'], 0.5)
            matched_terms['risk'].append('suppressed:broad/context-only risk language')

    scores = {
        'opportunity_score': score_payload['opportunity'],
        'implementation_score': score_payload['implementation'],
        'risk_score': score_payload['risk'],
    }

    ranked = sorted(score_payload.items(), key=lambda item: item[1], reverse=True)
    top_category, top_score = ranked[0]
    second_score = ranked[1][1]
    margin = top_score - second_score

    clear_dominant = (
        has_anchor(sentence, top_category)
        and top_score >= VERY_CLEAR_SCORE
        and margin > 0
    )

    if top_score < MIN_DOMINANT_SCORE:
        assigned = 'mixed_unclear'
    elif margin < MIN_DOMINANCE_MARGIN and not clear_dominant:
        assigned = 'mixed_unclear'
    elif top_category == 'opportunity' and VAGUE_AI_MENTION_RE.search(sentence) and not has_anchor(sentence, 'opportunity'):
        assigned = 'mixed_unclear'
    elif top_category == 'risk' and not EXPLICIT_RISK_RE.search(sentence):
        assigned = 'mixed_unclear'
    else:
        assigned = top_category

    all_matched = []
    for category in ['opportunity', 'implementation', 'risk']:
        unique_terms = list(dict.fromkeys(matched_terms[category]))
        for term in unique_terms:
            all_matched.append(f'{category}:{term}')
    all_matched.append(f'rule:top={top_category}; margin={margin:.2f}; threshold={MIN_DOMINANT_SCORE}')
    return assigned, scores, '; '.join(all_matched)

## 5. Create sentence-level AI framing classifications

In [ ]:
sentence_records = []
processed_transcripts = 0

for _, row in matched.iterrows():
    text = row.get('transcript_text_full')
    if pd.isna(text):
        continue
    sentences = split_sentences(text)
    processed_transcripts += 1
    for idx, sentence in enumerate(sentences, start=1):
        terms = matched_ai_terms(sentence)
        if not terms:
            continue
        prev_sentence = sentences[idx - 2] if idx > 1 else ''
        next_sentence = sentences[idx] if idx < len(sentences) else ''
        assigned, scores, framing_terms = classify_ai_sentence(prev_sentence, sentence, next_sentence)
        sentence_records.append({
            'ticker': row.get('ticker', pd.NA),
            'companyname': row.get('companyname', pd.NA),
            'transcriptid': row.get('transcriptid', pd.NA),
            'keydevid': row.get('keydevid', pd.NA),
            'call_date': row.get('call_date', pd.NA),
            'sentence_id': idx,
            'sentence_text': sentence,
            'matched_ai_terms': '; '.join(terms),
            'assigned_framing': assigned,
            'opportunity_score': scores['opportunity_score'],
            'implementation_score': scores['implementation_score'],
            'risk_score': scores['risk_score'],
            'matched_framing_terms': framing_terms,
            '_row_id': row.get('_row_id'),
        })

sentence_df = pd.DataFrame(sentence_records)
expected_sentence_cols = [
    'ticker', 'companyname', 'transcriptid', 'keydevid', 'call_date', 'sentence_id',
    'sentence_text', 'matched_ai_terms', 'assigned_framing', 'opportunity_score',
    'implementation_score', 'risk_score', 'matched_framing_terms'
]

if sentence_df.empty:
    sentence_df = pd.DataFrame(columns=expected_sentence_cols + ['_row_id'])

sentence_df[expected_sentence_cols].to_csv(SENTENCE_OUTPUT_PATH, index=False)
print(f'Saved sentence-level classification file: {SENTENCE_OUTPUT_PATH}')
print(f'Total AI-related sentences: {len(sentence_df):,}')
print(sentence_df['assigned_framing'].value_counts(dropna=False) if not sentence_df.empty else 'No AI sentences found')
sentence_df.head(10)

## 6. Aggregate sentence-level results to earnings-call level

In [ ]:
framing_categories = ['opportunity', 'implementation', 'risk', 'mixed_unclear']

if sentence_df.empty:
    agg = pd.DataFrame({'_row_id': matched['_row_id']})
    for cat in framing_categories:
        agg[f'{cat}_sentence_count'] = 0
else:
    counts = (
        sentence_df
        .pivot_table(index='_row_id', columns='assigned_framing', values='sentence_id', aggfunc='count', fill_value=0)
        .reset_index()
    )
    agg = pd.DataFrame({'_row_id': matched['_row_id']}).merge(counts, on='_row_id', how='left')
    for cat in framing_categories:
        if cat not in agg.columns:
            agg[cat] = 0
        agg[f'{cat}_sentence_count'] = agg[cat].fillna(0).astype(int)
    agg = agg[['_row_id'] + [f'{cat}_sentence_count' for cat in framing_categories]]

expanded = matched.merge(agg, on='_row_id', how='left')
for cat in framing_categories:
    expanded[f'{cat}_sentence_count'] = expanded[f'{cat}_sentence_count'].fillna(0).astype(int)

expanded['ai_sentence_count_framing'] = expanded[[f'{cat}_sentence_count' for cat in framing_categories]].sum(axis=1)

# Keep the original total_sentences if available; otherwise use sentence splitter counts from matched text.
if 'total_sentences' in expanded.columns:
    denominator = pd.to_numeric(expanded['total_sentences'], errors='coerce')
else:
    denominator = expanded['transcript_text_full'].apply(lambda x: len(split_sentences(x)) if pd.notna(x) else np.nan)

denominator = denominator.replace({0: np.nan})
for cat in ['opportunity', 'implementation', 'risk']:
    expanded[f'{cat}_intensity'] = expanded[f'{cat}_sentence_count'] / denominator
    expanded[f'{cat}_share'] = np.where(
        expanded['ai_sentence_count_framing'] > 0,
        expanded[f'{cat}_sentence_count'] / expanded['ai_sentence_count_framing'],
        np.nan
    )

expanded['mixed_unclear_share'] = np.where(
    expanded['ai_sentence_count_framing'] > 0,
    expanded['mixed_unclear_sentence_count'] / expanded['ai_sentence_count_framing'],
    np.nan
)

# Replace intensity missing values caused by missing/zero total sentence denominators with 0 when the count is 0.
for cat in ['opportunity', 'implementation', 'risk']:
    expanded[f'{cat}_intensity'] = expanded[f'{cat}_intensity'].fillna(0)

helper_cols = [
    '_row_id', '_transcriptid_key', '_keydevid_key', '_ticker_key', '_headline_key', '_call_date_key',
    'transcript_text_full', 'text_match_method'
]
output_cols = [c for c in expanded.columns if c not in helper_cols]
expanded_output = expanded[output_cols].copy()

# Place new framing variables near the original AI variables when possible.
new_cols = [
    'opportunity_sentence_count', 'implementation_sentence_count', 'risk_sentence_count', 'mixed_unclear_sentence_count',
    'opportunity_intensity', 'implementation_intensity', 'risk_intensity',
    'opportunity_share', 'implementation_share', 'risk_share', 'mixed_unclear_share'
]
base_cols = [c for c in expanded_output.columns if c not in new_cols]
expanded_output = expanded_output[base_cols + new_cols]
expanded_output.to_csv(FRAMING_DATASET_OUTPUT_PATH, index=False)

print(f'Saved expanded call-level framing dataset: {FRAMING_DATASET_OUTPUT_PATH}')
print(f'Rows: {len(expanded_output):,}; Columns: {len(expanded_output.columns):,}')
expanded_output[new_cols].describe().T

## 7. Create validation sample

Randomly samples up to 50 sentences from each main category using `random_state=42`.

In [ ]:
validation_parts = []
for category in ['opportunity', 'implementation', 'risk']:
    category_df = sentence_df[sentence_df['assigned_framing'] == category]
    n = min(50, len(category_df))
    if n > 0:
        validation_parts.append(category_df.sample(n=n, random_state=42))

validation_cols = [
    'ticker', 'companyname', 'call_date', 'sentence_text', 'assigned_framing',
    'opportunity_score', 'implementation_score', 'risk_score', 'matched_framing_terms'
]

if validation_parts:
    validation_df = pd.concat(validation_parts, ignore_index=True)[validation_cols]
else:
    validation_df = pd.DataFrame(columns=validation_cols)

validation_df['manual_check'] = ''
validation_df['manual_comment'] = ''
validation_df.to_csv(VALIDATION_OUTPUT_PATH, index=False)

print(f'Saved validation sample: {VALIDATION_OUTPUT_PATH}')
print(validation_df['assigned_framing'].value_counts(dropna=False) if not validation_df.empty else 'No validation rows sampled')
validation_df.head(10)

## 8. Summary diagnostics

In [ ]:
processed = int(processed_transcripts)
matched_with_text = int(matched['transcript_text_full'].notna().sum())
total_ai_sentences = int(len(sentence_df))
zero_ai_calls = int((expanded['ai_sentence_count_framing'] == 0).sum())

print('Summary diagnostics')
print('-------------------')
print(f'Number of transcripts processed: {processed:,}')
print(f'Number of transcripts matched with full text: {matched_with_text:,}')
print(f'Total AI-related sentences: {total_ai_sentences:,}')

for category in ['opportunity', 'implementation', 'risk', 'mixed_unclear']:
    count = int((sentence_df['assigned_framing'] == category).sum()) if not sentence_df.empty else 0
    share = count / total_ai_sentences if total_ai_sentences else 0
    label = category.replace('_', ' ')
    print(f'Number and share of {label} sentences: {count:,} ({share:.2%})')

print(f'Number of earnings calls with zero AI-related sentences: {zero_ai_calls:,}')
print('\nOutput files:')
print(f'- {SENTENCE_OUTPUT_PATH}')
print(f'- {VALIDATION_OUTPUT_PATH}')
print(f'- {FRAMING_DATASET_OUTPUT_PATH}')